In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 2. Análisis de bloques de ocupación (CON TIEMPO)
# DESCRIPCIÓN:
#   Extrae y analiza bloques continuos de ocupación a partir de datos
#   semanales del sensor del pasillo (step 1). Identifica períodos con
#   niveles constantes de ocupación y calcula estadísticas de variables
#   ambientales dentro de cada bloque.
#   NOVEDAD: Añade Minutes_Since_Start, Minutes_Since_First_Occupancy y *_trend
# ===================================================================

import pandas as pd
import numpy as np
import os
from openpyxl.styles import PatternFill, Font
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = '.'
OUTPUT_DIR = 'occupancy_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']

# ===================================================================
# FUNCIÓN ACTUALIZADA CON VARIABLES DE TIEMPO
# ===================================================================
def extract_occupancy_blocks_with_time(df_week, week_name):
    """
    Extrae bloques de ocupación añadiendo:
    - Minutes_Since_Start: minutos desde el inicio del bloque actual
    - Minutes_Since_First_Occupancy: minutos desde la primera ocupación del día
    - *_trend: tendencia del sensor durante el bloque (cambio/minuto)
    """
    occ_row = df_week[df_week['Variable_Measure'] == 'Occupancy']
    if occ_row.empty:
        return []

    occ_row = occ_row.iloc[0]
    date_cols = sorted([c for c in df_week.columns if c not in metadata_cols])

    # Construir secuencia de ocupación
    occupancy_sequence = []
    for col in date_cols:
        val = occ_row[col]
        if pd.notna(val):
            try:
                occupancy_sequence.append({
                    'DateTime': pd.to_datetime(col),
                    'Occupancy': int(val)
                })
            except:
                continue

    if not occupancy_sequence:
        return []

    # Encontrar la primera ocupación del día (>0)
    first_occ_dt = None
    for entry in occupancy_sequence:
        if entry['Occupancy'] > 0:
            first_occ_dt = entry['DateTime']
            break

    # Agrupar en bloques por cambio de ocupación
    blocks = []
    current_block = [occupancy_sequence[0]]
    for i in range(1, len(occupancy_sequence)):
        if occupancy_sequence[i]['Occupancy'] != occupancy_sequence[i-1]['Occupancy']:
            blocks.append(current_block)
            current_block = [occupancy_sequence[i]]
        else:
            current_block.append(occupancy_sequence[i])
    blocks.append(current_block)

    sensor_data = df_week[~df_week['Variable_Measure'].isin(['Occupancy', 'Door_Open', 'Window_Open'])]
    date_cols_dt = [pd.to_datetime(c) for c in date_cols]

    rows = []
    for block in blocks:
        block_start = block[0]['DateTime']
        block_end   = block[-1]['DateTime']
        block_indices = [i for i, dt in enumerate(date_cols_dt) if block_start <= dt <= block_end]

        if not block_indices:
            continue

        # ===== CALCULAR VARIABLES DE TIEMPO =====
        minutes_since_start = round((block_end - block_start).total_seconds() / 60)

        if first_occ_dt is not None and block_start >= first_occ_dt:
            minutes_since_first_occ = round((block_start - first_occ_dt).total_seconds() / 60)
        else:
            minutes_since_first_occ = 0

        row = {
            'Occupancy':                     block[0]['Occupancy'],
            'Date':                          block_start.date(),
            'Day_of_Week':                   block_start.strftime('%A'),
            'Start_Time':                    block_start.time(),
            'End_Time':                      block_end.time(),
            'Duration_min':                  minutes_since_start,
            'Minutes_Since_Start':           minutes_since_start,               # ← NUEVO
            'Minutes_Since_First_Occupancy': minutes_since_first_occ,           # ← NUEVO
            'Week':                          week_name
        }

        # Procesar datos de sensores con tendencia
        for _, sensor_row in sensor_data.iterrows():
            values = []
            for idx in block_indices:
                val = sensor_row[date_cols[idx]]
                if pd.notna(val):
                    values.append(float(val))

            if values:
                prefix = f"{sensor_row['Sensor_Location']}_{sensor_row['Variable_Measure']}" \
                         if sensor_row['Sensor_Location'] else sensor_row['Variable_Measure']

                # Calcular tendencia (cambio por minuto)
                if len(values) > 1:
                    time_span_minutes = (date_cols_dt[block_indices[-1]] - date_cols_dt[block_indices[0]]).total_seconds() / 60
                    if time_span_minutes > 0:
                        trend = (values[-1] - values[0]) / time_span_minutes
                    else:
                        trend = 0
                else:
                    trend = 0

                row.update({
                    f'{prefix}_mean':  np.mean(values),
                    f'{prefix}_std':   np.std(values) if len(values) > 1 else 0,
                    f'{prefix}_max':   np.max(values),
                    f'{prefix}_min':   np.min(values),
                    f'{prefix}_trend': round(trend, 4),  # ← NUEVO: tendencia
                })

        rows.append(row)

    return rows


# ==========================================
# CARGAR EXCEL SEMANALES Y EXTRAER BLOQUES
# ==========================================
weekly_files = sorted([f for f in os.listdir(INPUT_DIR) if f.startswith('1.5_data_') and f.endswith('.xlsx')])
print(f"Encontrados {len(weekly_files)} archivos semanales")

all_occupancy_rows = []
for fname in weekly_files:
    print(f"   Procesando {fname}...")
    df_week = pd.read_excel(os.path.join(INPUT_DIR, fname))
    week_name = fname.replace('1.5_data_', '').replace('.xlsx', '')
    
    # ===== USAR LA NUEVA FUNCIÓN =====
    rows = extract_occupancy_blocks_with_time(df_week, week_name)
    all_occupancy_rows.extend(rows)
    print(f"      Bloques encontrados: {len(rows)}")

# ==========================================
# GUARDAR RESULTADOS
# ==========================================
if all_occupancy_rows:
    df_occupancy = pd.DataFrame(all_occupancy_rows)

    # Reordenar columnas: tiempo primero, luego sensores
    first_cols = ['Occupancy', 'Date', 'Day_of_Week', 'Start_Time', 'End_Time',
                  'Duration_min', 'Minutes_Since_Start', 'Minutes_Since_First_Occupancy', 'Week']
    other_cols = sorted([c for c in df_occupancy.columns if c not in first_cols])
    df_occupancy = df_occupancy[first_cols + other_cols].sort_values(['Date', 'Start_Time'])

    file_final = os.path.join(OUTPUT_DIR, '1.5_occupancy_blocks_analysis.xlsx')
    with pd.ExcelWriter(file_final, engine='openpyxl') as writer:
        df_occupancy.to_excel(writer, sheet_name='Occupancy_Blocks', index=False)
        ws = writer.sheets['Occupancy_Blocks']

        for cell in ws[1]:
            cell.fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
            cell.font = Font(bold=True, size=11, color='FFFFFF')

        occ_col = list(df_occupancy.columns).index('Occupancy') + 1
        colors = {0: 'F2F2F2', 10: 'E2EFDA', 30: 'FCE4D6'}
        for row in range(2, ws.max_row + 1):
            val = ws.cell(row=row, column=occ_col).value
            if val is not None:
                for threshold, color in sorted(colors.items()):
                    if val < threshold or threshold == max(colors.keys()):
                        if val < threshold:
                            break
                color = next((c for t, c in sorted(colors.items()) if val < t), 'F4B4C2')
                ws.cell(row=row, column=occ_col).fill = PatternFill(start_color=color, end_color=color, fill_type='solid')

    print(f"\nGuardado en: {file_final}")
    print(f"Total de bloques: {len(df_occupancy)}")
    print(f"Niveles de ocupación: {sorted(df_occupancy['Occupancy'].unique())}")
    print(f"\nColumnas generadas ({len(df_occupancy.columns)}):")
    for col in df_occupancy.columns:
        marca = " 🕐" if "Minutes" in col or "Since" in col else (" 📈" if "_trend" in col else "")
        print(f"   - {col}{marca}")
else:
    print("No se encontraron bloques de ocupación.")

print("\n¡Proceso completado!")

Encontrados 12 archivos semanales
   Procesando 1.5_data_December_2025_week_4_days_22-31.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_February_2026_week_1_days_1-7.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_February_2026_week_2_days_8-14.xlsx...
      Bloques encontrados: 38
   Procesando 1.5_data_February_2026_week_3_days_15-21.xlsx...
      Bloques encontrados: 12
   Procesando 1.5_data_February_2026_week_4_days_22-28.xlsx...
      Bloques encontrados: 13
   Procesando 1.5_data_January_2026_week_1_days_1-7.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_January_2026_week_2_days_8-14.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_January_2026_week_3_days_15-21.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_January_2026_week_4_days_22-31.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_March_2026_week_1_days_1-7.xlsx...
      Bloques encontrados: 29
   Procesando 1.5_data_March_2026_week_2_days_8-14.xlsx..